In [ ]:
# 1. Монтируем Google Диск
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# 2. Разархивируем сохраненную модель
!unzip /content/drive/MyDrive/empathy_checkpoints_ru/manual-save.zip -d /content/empathy_model


Archive:  /content/drive/MyDrive/empathy_checkpoints_ru/manual-save.zip
  inflating: /content/empathy_model/special_tokens_map.json  
  inflating: /content/empathy_model/model.safetensors.index.json  
  inflating: /content/empathy_model/model-00003-of-00003.safetensors  
  inflating: /content/empathy_model/added_tokens.json  
  inflating: /content/empathy_model/tokenizer.model  
  inflating: /content/empathy_model/model-00002-of-00003.safetensors  
  inflating: /content/empathy_model/training_args.bin  
  inflating: /content/empathy_model/tokenizer_config.json  
  inflating: /content/empathy_model/config.json  
  inflating: /content/empathy_model/model-00001-of-00003.safetensors  
  inflating: /content/empathy_model/generation_config.json  
  inflating: /content/empathy_model/tokenizer.json  


In [ ]:
# 3. Установка нужных библиотек
!pip install transformers peft accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [ ]:
ls -lh /content/empathy_model/*.safetensors


-rw-r--r-- 1 root root 4.1G May  7 22:52 /content/empathy_model/model-00001-of-00003.safetensors
-rw-r--r-- 1 root root 4.7G May  7 22:47 /content/empathy_model/model-00002-of-00003.safetensors
-rw-r--r-- 1 root root 4.3G May  7 22:48 /content/empathy_model/model-00003-of-00003.safetensors


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/empathy_model"

# 1. Загрузим токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 2. Загрузим конфиг
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_path)

# 3. Создаем модель по конфигу
model = AutoModelForCausalLM.from_config(config)

# 4. Загружаем веса вручную
import os
from safetensors.torch import load_file

# Собираем все веса из шардов
state_dict = {}
for i in range(1, 4):
    shard_path = os.path.join(model_path, f"model-0000{i}-of-00003.safetensors")
    shard = load_file(shard_path)
    state_dict.update(shard)

model.load_state_dict(state_dict)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()


In [ ]:
# 5. Проверка: ввод тестовых реплик
def chat(prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.95,
            temperature=0.7,
        )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Пример использования
chat("Мне сегодня очень грустно и одиноко. Что ты можешь сказать?")
chat("Я устал от всего. Что делать?")
chat("Сегодня был хороший день, я чувствую себя счастливым.")
